In [2]:
!pip install pandas requests

In [3]:
# Import modules that will be used to retrieve data from the InterPro API
import requests
import json

In [4]:
url = "https://www.ebi.ac.uk/interpro/api/Protein"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    data = response.json()
    print(json.dumps(data, indent=4)) # Use indent=4 for pretty printing with line breaks
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

{
    "proteins": {
        "unreviewed": 251395512,
        "reviewed": 570270,
        "uniprot": 251743096
    }
}


In [5]:
url = "https://www.ebi.ac.uk/interpro/api/protein/uniprot/Q7A5Z7"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    data = response.json()
    print(json.dumps(data, indent=4)) # Use indent=4 for pretty printing with line breaks
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

Error fetching data: Expecting value: line 1 column 1 (char 0)


In [6]:
url = "https://www.ebi.ac.uk/interpro/api/protein/uniprot/Q7Z5A7"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    data = response.json()
    print(json.dumps(data, indent=4)) # Use indent=4 for pretty printing with line breaks
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

{
    "metadata": {
        "accession": "Q7Z5A7",
        "id": "TAFA5_HUMAN",
        "source_organism": {
            "taxId": "9606",
            "scientificName": "Homo sapiens",
            "fullName": "Homo sapiens (Human)"
        },
        "name": "Chemokine-like protein TAFA-5",
        "description": [
            "Acts as a chemokine-like protein by regulating cell proliferation and migration through activation of G protein-coupled receptors (GPCRs), such as S1PR2 and FPR2 (By similarity). Stimulates chemotactic migration of macrophages mediated by the MAPK3/ERK1 and AKT1 pathway (By similarity). Blocks TNFSF11/RANKL-induced osteoclast formation from macrophages by inhibiting up-regulation of osteoclast fusogenic and differentiation genes (By similarity). Stimulation of macrophage migration and inhibition of osteoclast formation is mediated via GPCR FPR2 (By similarity). Acts as an adipokine by negatively regulating vascular smooth muscle cell (VSMC) proliferation and migr

In [7]:
import json, re
import requests
from time import sleep
import pandas as pd
from IPython.display import display

In [8]:
INTERPRO_API_BASE = "https://www.ebi.ac.uk/interpro/api/"

In [9]:
def get_interpro_data(url):
  output = []

  last_page = False

  while url:
    attempts = 0
    while attempts < 3:
      try:
        response = requests.get(url, headers={"Accept": "application/json"})
        if response.status_code == 408:
          attempts += 1
          print(f"Received 408 Timeout. Retrying {attempts}/3...")
          sleep(61)
          continue
        elif response.status_code == 204:
          #no data so leave loop
          break
        response.raise_for_status()
        data = response.json()

        if data.get('results'):
          output.extend(data.get('results'))
        else:
          output.append(data)

        url = data.get('next', "")
        sleep(1)
        break
      except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        break
    else:
      print("Max retries reached for URL:", url)
      break

  return output

In [10]:
def get_protein_annotations(results):
  if not results:
    return pd.DataFrame(columns=["InterPro ID", "Description", "Type", "Location"])

  records = []
  for entry in results:
    interpro_id = entry.get("metadata", {}).get("accession", "N/A")
    description = entry.get("metadata", {}).get("name", "N/A")
    entry_type = entry.get("metadata", {}).get("type", "N/A")
    location_strings = []
    for protein_entry in entry.get("proteins", []): # Iterate through protein entries
      for location in protein_entry.get("entry_protein_locations", []): # Iterate through location entries
        for fragment in location.get("fragments", []): # Iterate through fragments
          start = fragment.get("start", "N/A")
          end = fragment.get("end", "N/A")
          location_strings.append(f"{start}-{end}")

    location_str = ",".join(location_strings) if location_strings else "N/A"

    records.append([interpro_id, description, entry_type, location_str])

  return pd.DataFrame(records, columns=["InterPro ID", "Description", "Type", "Location"])

In [11]:
uniprot_id = "Q7Z5A7"

main_data_type = "entry/InterPro"
secondary_data_type = f"protein/UniProt/{uniprot_id}"
url = f"{INTERPRO_API_BASE}{main_data_type}/{secondary_data_type}/?page_size=200"
print(f"InterPro API url: {url}")

# Get the data
interpro_results = get_interpro_data(url)
interpro_df = get_protein_annotations(interpro_results)

# display the DataFrame in a table format
display(interpro_df)

InterPro API url: https://www.ebi.ac.uk/interpro/api/entry/InterPro/protein/UniProt/Q7Z5A7/?page_size=200


,InterPro ID,Description,Type,Location
0,IPR020350,Chemokine-like protein TAFA,family,42-131
1,IPR040329,Chemokine-like protein TAFA-5,family,5-131


In [24]:
# Print the DataFrame in TSV format
print(interpro_df.to_csv(sep='\t', index=False, header=True))

Accession	Name	Organism	Domain Count
A0A060W5B1	Uncharacterized protein	N/A	0
A0A060W5X8	TAFA chemokine like family member 5	N/A	0
A0A060X457	TAFA chemokine like family member 5	N/A	0
A0A060Y997	TAFA chemokine like family member 5	N/A	0
A0A060YA69	TAFA chemokine like family member 5	N/A	0
A0A087XVE2	TAFA chemokine like family member 5a	N/A	0
A0A087YGK7	TAFA chemokine like family member 5	N/A	0
A0A091F6L5	Protein FAM19A5	N/A	0
A0A091GI63	Protein FAM19A5	N/A	0
A0A091IKQ7	Protein FAM19A5	N/A	0
A0A091K5J3	Protein FAM19A5	N/A	0
A0A091N3U6	Protein FAM19A5	N/A	0
A0A091RS92	Protein FAM19A5	N/A	0
A0A091UX58	Protein FAM19A5	N/A	0
A0A093NW58	Protein FAM19A5	N/A	0
A0A093Q321	Protein FAM19A5	N/A	0
A0A099ZI82	Protein FAM19A5	N/A	0
A0A0A0AED6	Protein FAM19A5	N/A	0
A0A0D9QYL2	TAFA chemokine like family member 5	N/A	0
A0A0N8EUF2	Protein FAM19A5 isoform 1	N/A	0
A0A0P7U0D6	Protein FAM19A5-like	N/A	0
A0A0P7U8N8	Protein FAM19A5-like	N/A	0
A0A0Q3R696	Protein FAM19A5	N/A	0
A0A147ABK4	Protein FAM19A5	N/A	0
A0

In [12]:
def get_interpro_entry_info(results):
  if not results:
    return pd.DataFrame(columns=["InterPro ID", "Description Text", "Description LLM", "Description Checked", "GO Terms"])

  records = []
  for entry in results:
    interpro_id = entry.get("metadata", {}).get("accession", "N/A")

    go_terms = []
    # Extract GO terms and format them (this is done once per entry)
    go_term_list = entry.get("metadata", {}).get("go_terms", [])
    if go_term_list:
      for go in go_term_list:
        go_id=go.get("identifier", "N/A")
        go_name=go.get("name", "N/A")
        go_terms.append(f"{go_id}={go_name}")

    # Combine GO terms into a single string
    go_terms_str = ','.join(go_terms) if go_terms else "N/A"

    descriptions = entry.get("metadata", {}).get("description", [])

    # If there are no descriptions, create one row with N/A for description columns
    if not descriptions:
      records.append([interpro_id, "N/A", "N/A", "N/A", go_terms_str])
    else:
      # If there are descriptions, create a row for each description
      for desc in descriptions:
        description_text_raw = desc.get("text", "N/A")
        description_llm = desc.get("llm", "N/A")
        description_checked = desc.get("checked", "N/A")

        # Reformat description_text_raw to remove HTML paragraph tags
        # Use re.sub to replace <p> or </p> with a newline character
        # The | is the OR operator in regex
        description_text_formatted = re.sub(r'<p>|</p>', '\n', description_text_raw)
        # Optionally, remove any leading/trailing whitespace that might result from replacement
        description_text_formatted = description_text_formatted.strip()

        # Append a row for each description, repeating InterPro ID and GO Terms
        records.append([interpro_id, description_text_formatted, description_llm, description_checked, go_terms_str])

  # Create and return the DataFrame with updated column names
  return pd.DataFrame(records, columns=["InterPro ID", "Description Text", "Description LLM", "Description Checked", "GO Terms"])

In [13]:
url = ""
print(url)

# Get the data
entry_results = get_interpro_data(url)
entry_df = get_interpro_entry_info(entry_results)

# display the DataFrame in a table format
display(entry_df)

,InterPro ID,Description Text,Description LLM,Description Checked,GO Terms


In [14]:

import requests
import pandas as pd

# UniProt ID for Fam19A5
uniprot_id = "Q7Z5A7"

# Base API for InterPro
INTERPRO_API_BASE = "https://www.ebi.ac.uk/interpro/api/"

# Construct the URL for protein entries
main_data_type = f"protein/uniprot/{uniprot_id}"
url = f"{INTERPRO_API_BASE}{main_data_type}"

print("Fetching data from:", url)

# Get the data from InterPro API
response = requests.get(url)
if response.status_code == 200:
    entry_results = response.json()
else:
    raise Exception(f"Error fetching data: {response.status_code}")

# Convert JSON to DataFrame (flatten relevant fields)
def get_interpro_entry_info(entry_results):
    data = []
    if "metadata" in entry_results:
        data.append({
            "Protein Name": entry_results["metadata"].get("name"),
            "Organism": entry_results["metadata"].get("source_organism", {}).get("scientificName"),
            "Length": entry_results["metadata"].get("length"),
            "Accession": entry_results["metadata"].get("accession")
        })
    return pd.DataFrame(data)

# Create DataFrame
entry_df = get_interpro_entry_info(entry_results)

# Display table
from IPython.display import display
display(entry_df)

Fetching data from: https://www.ebi.ac.uk/interpro/api/protein/uniprot/Q7Z5A7


,Protein Name,Organism,Length,Accession
0,Chemokine-like protein TAFA-5,Homo sapiens,132,Q7Z5A7


In [15]:
def get_representative_domains(results):
  if not results:
    return pd.DataFrame(columns=["Member db", "Description", "Type", "Location"])

  records = []
  representative_types = ["domain", "homologous_superfamily", "repeat"]
  for entry in results:
    member_db = entry.get("metadata", {}).get("source_database", "N/A")
    description = entry.get("metadata", {}).get("name", "N/A")
    entry_type = entry.get("metadata", {}).get("type", "N/A")

    if entry_type in representative_types:
      location_strings = []
      for protein_entry in entry.get("proteins", []):
        for location in protein_entry.get("entry_protein_locations", []):
          # Check if the location is representative
          if location.get("representative", False):
            for fragment in location.get("fragments", []):
                start = fragment.get("start", "N/A")
                end = fragment.get("end", "N/A")
                location_strings.append(f"{start}-{end}")

      location_str = ",".join(location_strings) if location_strings else "N/A"

      # Only append if representative locations were found
      if location_str != "N/A":
          records.append([member_db, description, entry_type, location_str])

  return pd.DataFrame(records, columns=["Member db", "Description", "Type", "Location"])

In [25]:
url = ""
print(url)

# Get the data
entry_results = get_interpro_data(url)
entry_df = get_interpro_entry_info(entry_results)

# display the DataFrame in a table format
display(entry_df)

""


In [26]:
#@title Answer

entry_id = "IPR040329"
main_data_type =f"entry/InterPro/{entry_id}"
url = f"{INTERPRO_API_BASE}{main_data_type}"
print(url)

# Get the data
entry_results = get_interpro_data(url)
entry_df = get_interpro_entry_info(entry_results)

# display the DataFrame in a table format
display(entry_df)

https://www.ebi.ac.uk/interpro/api/entry/InterPro/IPR040329


""


In [27]:
def get_representative_domains(results):
  if not results:
    return pd.DataFrame(columns=["Member db", "Description", "Type", "Location"])

  records = []
  representative_types = ["domain", "homologous_superfamily", "repeat"]
  for entry in results:
    member_db = entry.get("metadata", {}).get("source_database", "N/A")
    description = entry.get("metadata", {}).get("name", "N/A")
    entry_type = entry.get("metadata", {}).get("type", "N/A")

    if entry_type in representative_types:
      location_strings = []
      for protein_entry in entry.get("proteins", []):
        for location in protein_entry.get("entry_protein_locations", []):
          # Check if the location is representative
          if location.get("representative", False):
            for fragment in location.get("fragments", []):
                start = fragment.get("start", "N/A")
                end = fragment.get("end", "N/A")
                location_strings.append(f"{start}-{end}")

      location_str = ",".join(location_strings) if location_strings else "N/A"

      # Only append if representative locations were found
      if location_str != "N/A":
          records.append([member_db, description, entry_type, location_str])

  return pd.DataFrame(records, columns=["Member db", "Description", "Type", "Location"])

In [16]:
uniprot_id = "Q7Z5A7"
main_data_type = f"entry/all"
second_data_type = f"protein/UniProt/{uniprot_id}"
url = f"{INTERPRO_API_BASE}{main_data_type}/{second_data_type}/?page_size=200"
print(f"InterPro API url: {url}")

# Get the data
interpro_results = get_interpro_data(url)
representative_domains_df = get_representative_domains(interpro_results)

# # display the DataFrame in a table format
display(representative_domains_df)

InterPro API url: https://www.ebi.ac.uk/interpro/api/entry/all/protein/UniProt/Q7Z5A7/?page_size=200


,Member db,Description,Type,Location


In [17]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
interpro_id = "IPR040329"   # InterPro entry for Chemokine-like protein TAFA-5 (Fam19A5)
base_url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/entry/interpro/{interpro_id}/"

In [19]:
# reviewed:true (Swiss-Prot), structure_db:AlphaFold
params = {
    "extra_fields": "organism",
    "filters": "reviewed:true,structure_db:AlphaFold",
    "size": 200  # results per page
}

In [20]:
# -------- DATA COLLECTION --------
results = []
url = base_url
while url:
    r = requests.get(url, params=params, headers={"Accept": "application/json"})
    if not r.ok:
        print("Error:", r.status_code, r.text)
        break

    data = r.json()

    for item in data["results"]:
        accession = item["metadata"].get("accession", "N/A")
        name = item["metadata"].get("name", "N/A")
        # Safely access the organism name
        organism_data = item["metadata"].get("organism")
        organism = organism_data.get("scientificName", "N/A") if organism_data else "N/A"

        # Count how many matching domains
        # The structure might vary, so handle cases where entry_protein_locations might be missing
        domain_count = len(item.get("entry_protein_locations", []))

        results.append([accession, name, organism, domain_count])

    # handle pagination
    url = data["next"] if "next" in data and data["next"] else None

Error: 400 <html>
  <head>
    <title>Bad Request</title>
  </head>
  <body>
    <h1><p>Bad Request</p></h1>
    Request Line is too large (4108 &gt; 4094)
  </body>
</html>



In [21]:
# -------- DATA PROCESSING --------
# Create a DataFrame
interpro_df = pd.DataFrame(results, columns=["Accession", "Name", "Organism", "Domain Count"])

# -------- DATA DISPLAY --------
# Display the DataFrame
display(interpro_df)

,Accession,Name,Organism,Domain Count
0,A0A060W5B1,Uncharacterized protein,N/A,0
1,A0A060W5X8,TAFA chemokine like family member 5,N/A,0
2,A0A060X457,TAFA chemokine like family member 5,N/A,0
3,A0A060Y997,TAFA chemokine like family member 5,N/A,0
4,A0A060YA69,TAFA chemokine like family member 5,N/A,0
...,...,...,...,...
955,A0A8C7S4Z8,TAFA chemokine like family member 5,N/A,0
956,A0A8C7TUQ7,TAFA chemokine like family member 5,N/A,0
957,A0A8C7TW69,Uncharacterized protein,N/A,0
958,A0A8C7U2Z6,TAFA chemokine like family member 5,N/A,0
